# Compile State Stats

Stage 2 of the data pipeline. Reads the CSVs produced by `pipeline.ipynb` from `output_csvs/`, 
filters each to the latest year and residential sector where applicable, 
and joins everything into a single state-level summary table (`output_csvs/state_stats.csv`).

In [1]:
import os
import pandas as pd
import sys
sys.path.append("../modules")

from dotenv import load_dotenv
load_dotenv("../.env")

from drive_uploader import get_drive_service, ensure_path, upload_df_to_drive
from solar_bill_savings_update import load_from_export
from solar_eligible_households import compute_solar_eligibility, apply_rooftop_suitability
from median_interconnection_timelines import run_pipeline as ix_run_pipeline

ROOT_ID = "1DBlVUvspIPTTyZPtVovYtEgUSmQNXmG7"
service = get_drive_service()

name_abbr = pd.read_csv("../data/state_name_abbr.csv")


In [2]:
# Bill savings (2026 cohort, baseline scenario)
savings = load_from_export("../data/bill_savings_csvs")
savings = savings.rename(columns={"state_abbr": "state"})
savings = savings[["state", "year_1_savings", "lifetime_savings"]]


In [3]:
# Residential electricity prices (from output_csvs, latest year, residential sector)
elec = pd.read_csv("../output_csvs/electricity_rates_by_state.csv")
elec = elec[elec["sector"] == "Residential"]
elec = elec[elec["year"] == elec["year"].max()]
elec = elec[["state", "price_per_kwh", "avg_annual_bill"]].rename(columns={"avg_annual_bill": "annual_bill"})

In [4]:
# Solar capacity/installations (from output_csvs, latest year, residential sector)
solar = pd.read_csv("../output_csvs/solar_storage_capacity_installations_by_state_sector.csv")
solar = solar[solar["sector"] == "Residential"]
solar = solar[solar["year"] == solar["year"].max()]
solar = solar[["state", "pv_customers", "pv_capacity_mw"]]

In [5]:
# Solar-eligible households (ResStock filter only)
# Rooftop suitability haircut is NOT applied in this run.
# To include it, uncomment the apply_rooftop_suitability line below.

resstock = pd.read_csv("../data/resstock_metadata_technical_potential.csv")
eligibility = compute_solar_eligibility(resstock)
# eligibility = apply_rooftop_suitability(eligibility)

In [6]:
# Residential PV technical generation potential by state (NREL SLOPE, 2020)
techpot = pd.read_csv("../output_csvs/potential_solar_generation_by_state.csv")


In [7]:
# Median solar cost and system size (LBNL Tracking the Sun, 2024)
solar_costs = pd.read_csv("../output_csvs/residential_solar_costs_by_state_over_time.csv")
solar_costs = solar_costs[solar_costs["year"] == 2024][["state", "median_price_per_kw", "median_system_size_kw"]]


In [8]:
# Cancellation rates (2022-2024 combined, min 500 applications)
cancel = pd.read_csv("../data/state_cancellation_rates.csv")
cancel = cancel[(cancel["year"] == "2022_2024") & (cancel["num_applications"] >= 500)]
cancel = cancel.merge(name_abbr, left_on="state", right_on="state", how="inner")
cancel = cancel[["state_abbr", "pct_cancelled"]].rename(columns={"state_abbr": "state"})

In [9]:
# Permitting timelines (Solar TRACE 2023, 0-10kW, min 250 installs)
_st_all = pd.read_csv("../output_csvs/solartrace_timelines_by_state.csv")

_st = _st_all[
    (_st_all["year"] == 2023) &
    (_st_all["size_class"] == "0_10kw") &
    (_st_all["installs"] >= 250)
]

solar_permit = (
    _st[_st["tech_class"] == "pv_only"][["state", "permit_time_days"]]
    .rename(columns={"permit_time_days": "median_permit_days_solar"})
)
storage_permit = (
    _st[_st["tech_class"] == "pv_storage"][["state", "permit_time_days"]]
    .rename(columns={"permit_time_days": "median_permit_days_solar_storage"})
)
permit_df = solar_permit.merge(storage_permit, on="state", how="outer")


In [10]:
# Interconnection timelines (Solar TRACE 2023, 0-10kW, min 250 installs)
# Total IX = pre-install IX + final IX to PTO
_st_ix = _st.copy()
_st_ix["total_ix_days"] = _st_ix["pre_install_ix_days"] + _st_ix["final_ix_to_pto_days"]

ix_pv = (
    _st_ix[_st_ix["tech_class"] == "pv_only"][["state", "total_ix_days"]]
    .rename(columns={"total_ix_days": "median_ix_days_pv_only"})
)
ix_storage = (
    _st_ix[_st_ix["tech_class"] == "pv_storage"][["state", "total_ix_days"]]
    .rename(columns={"total_ix_days": "median_ix_days_pv_storage"})
)
ix_df = ix_pv.merge(ix_storage, on="state", how="outer")


In [11]:
# Inspection timelines (Solar TRACE 2023, 0-10kW, min 250 installs)
insp_pv = (
    _st[_st["tech_class"] == "pv_only"][["state", "inspection_time_days"]]
    .rename(columns={"inspection_time_days": "median_inspection_days_pv_only"})
)
insp_storage = (
    _st[_st["tech_class"] == "pv_storage"][["state", "inspection_time_days"]]
    .rename(columns={"inspection_time_days": "median_inspection_days_pv_storage"})
)
insp_df = insp_pv.merge(insp_storage, on="state", how="outer")


In [12]:
# Join all state-level stats
state_stats = (
    savings
    .merge(elec, on="state", how="outer")
    .merge(solar, on="state", how="outer")
    .merge(eligibility, on="state", how="outer")
    .merge(techpot, on="state", how="outer")
    .merge(solar_costs, on="state", how="outer")
    .merge(cancel, on="state", how="outer")
    .merge(permit_df, on="state", how="outer")
    .merge(ix_df, on="state", how="outer")
    .merge(insp_df, on="state", how="outer")
)

# Drop rows not in state mapping (e.g. territories, US totals)
state_stats = state_stats.merge(name_abbr.rename(columns={"state": "state_name"}), left_on="state", right_on="state_abbr", how="inner")
state_stats = state_stats.drop(columns=["state_abbr"]).rename(columns={"state": "state_abbr"})

# Solar penetration: solar installations / eligible households
state_stats["solar_penetration_pct"] = state_stats["pv_customers"] / state_stats["eligible_households"]

# Rename columns for final output
state_stats = state_stats.rename(columns={
    "state_name":                       "State",
    "state_abbr":                       "State (Abbr.)",
    "year_1_savings":                   "Median solar savings in first year",
    "lifetime_savings":                 "Median solar savings over lifetime",
    "price_per_kwh":                    "Average electricity retail cost ($/kWh)",
    "annual_bill":                      "Average annual electricity bill",
    "pv_customers":                     "Number of solar installations",
    "pv_capacity_mw":                   "Total installed solar capacity (MW)",
    "total_households":                 "Total households",
    "eligible_households":              "Solar eligible and suitable households",
    "eligible_pct":                     "Percent of total households suitable",
    "solar_penetration_pct":            "Solar penetration",
    "technical_potential_gwh":          "Annual technical potential (GWh)",
    "median_price_per_kw":              "Median solar cost ($/kW)",
    "median_system_size_kw":            "Median solar system size (kW)",
    "pct_cancelled":                    "Solar permit cancellation rate (2022-2024)",
    "median_permit_days_solar":         "Median permitting timeline - solar (days)",
    "median_permit_days_solar_storage": "Median permitting timeline - solar+storage (days)",
    "median_ix_days_pv_only":               "Median interconnection timeline - PV only (days)",
    "median_ix_days_pv_storage":            "Median interconnection timeline - PV+storage (days)",
    "median_inspection_days_pv_only":       "Median inspection timeline - PV only (days)",
    "median_inspection_days_pv_storage":    "Median inspection timeline - PV+storage (days)",
})

# Final column order
state_stats = state_stats[[
    "State", "State (Abbr.)",
    "Median solar savings in first year", "Median solar savings over lifetime",
    "Average electricity retail cost ($/kWh)", "Average annual electricity bill",
    "Number of solar installations", "Total installed solar capacity (MW)",
    "Total households", "Solar eligible and suitable households",
    "Percent of total households suitable", "Solar penetration",
    "Annual technical potential (GWh)",
    "Median solar cost ($/kW)", "Median solar system size (kW)",
    "Solar permit cancellation rate (2022-2024)",
    "Median permitting timeline - solar (days)",
    "Median permitting timeline - solar+storage (days)",
    "Median interconnection timeline - PV only (days)",
    "Median interconnection timeline - PV+storage (days)",
    "Median inspection timeline - PV only (days)",
    "Median inspection timeline - PV+storage (days)",
]]

state_stats

,State,State (Abbr.),Median solar savings in first year,Median solar savings over lifetime,Average electricity retail cost ($/kWh),Average annual electricity bill,Number of solar installations,Total installed solar capacity (MW),Total households,Solar eligible and suitable households,...,Annual technical potential (GWh),Median solar cost ($/kW),Median solar system size (kW),Solar permit cancellation rate (2022-2024),Median permitting timeline - solar (days),Median permitting timeline - solar+storage (days),Median interconnection timeline - PV only (days),Median interconnection timeline - PV+storage (days),Median inspection timeline - PV only (days),Median inspection timeline - PV+storage (days)
0,Alaska,AK,NaN,NaN,0.265885,1783.753261,3011.0,16.227,3.186491e+05,1.508188e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Alabama,AL,953.758080,42926.396254,0.164076,2251.498253,0.0,4.458,2.301637e+06,1.128348e+06,...,10741.47579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Arkansas,AR,824.670133,36657.734014,0.130853,1676.893516,16021.0,180.206,1.398248e+06,6.769072e+05,...,6529.45901,NaN,NaN,0.1568,NaN,NaN,NaN,NaN,NaN,NaN
3,Arizona,AZ,2038.509747,91220.747140,0.156127,1909.552552,328287.0,2443.592,3.034657e+06,1.452583e+06,...,13993.68649,4046.826415,8.8000,0.1915,4.0,10.0,14.0,21.5,11.0,6.0
4,California,CA,1774.846810,81690.339880,0.331617,1844.891646,2195745.0,13795.564,1.449155e+07,6.632218e+06,...,70314.63613,4651.441885,5.8250,0.2313,4.0,9.0,12.0,18.0,11.0,12.0
5,Colorado,CO,1040.575022,48474.006062,0.161528,1305.111490,172555.0,1007.072,2.380093e+06,1.273073e+06,...,8612.71255,4401.575844,6.0000,0.1701,2.0,7.0,19.0,32.5,18.0,5.0
6,Connecticut,CT,1704.960285,77241.036971,0.299414,2495.812638,122781.0,934.825,1.556937e+06,8.759677e+05,...,5169.22788,5538.955219,7.0000,0.2196,6.0,NaN,18.0,NaN,12.0,NaN
7,District of Columbia,DC,NaN,NaN,0.223592,1707.298930,21479.0,141.148,3.196647e+05,8.937409e+04,...,156.98218,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Delaware,DE,1314.894999,58950.076292,0.174573,1945.776532,14054.0,114.279,4.354448e+05,2.338453e+05,...,1687.07653,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Florida,FL,1819.179594,81286.303092,0.155312,2030.249997,263388.0,2892.627,9.534083e+06,4.161227e+06,...,45803.02075,4558.918998,9.6150,0.2048,7.0,7.0,11.0,14.0,13.0,10.0


In [13]:
# Export to CSV (for copy-paste into Google Sheets)
state_stats.to_csv("../output_csvs/state_stats.csv", index=False)
print("Saved to output_csvs/state_stats.csv")

# # Upload to Google Drive
# folder_id = ensure_path(service, ROOT_ID, ["1. Solar"])
# upload_df_to_drive(state_stats, folder_id, "State Solar and Electricity Stats")

Saved to output_csvs/state_stats.csv
